# 第 44 课 · Agent 状态持久化与失败恢复

**目标：程序中断后继续同一个任务；既不遗失已经取得的结果，也不盲目重复可能已完成的写操作。**

第 43 课已经把多个工具调用与结果关联起来。本课将这些调用、消息和剩余预算落盘，并在不同中断点重启。

只使用 Python 3.10+ 标准库。没有模型、GPU、网络请求、Shell、真实工单或付款操作。`create_ticket` **只写入当前教学目录内的模拟服务数据库**。

本课固定策略用于隔离持久化机制，**不是模型自主规划成绩**。即使工作流完成，`task_success` 也保持 `None`，不自动宣称语义正确。

## 0. 学习顺序

先运行完整小案例，观察“状态已经保存到哪里”，再阅读对应实现，最后完成三个练习。实现较长的单元可以折叠后按需回看。

| 本课能力 | 验收问题 |
|---|---|
| 持久化状态 | 为什么只存聊天文本不够？ |
| 状态机 | `in_flight` 与 `failed` 有什么区别？ |
| 幂等重试 | 两次请求怎样只产生一次业务效果？ |
| 原子提交 | 完成标记与工具消息为什么要一起保存？ |
| 失败恢复 | 什么情况下必须停止，而不是重试？ |
| 实验验证 | 一个进程的异常恢复与断电测试是不是一回事？ |

本课的 Agent 检查点不是第 26 课的模型训练检查点，也不是长期用户记忆。它保存**工作流状态**，不保存模型参数、梯度或 KV cache。框架也会区分线程检查点与跨线程的长期 store。[LangGraph 持久化说明](https://docs.langchain.com/oss/python/langgraph/persistence)

In [1]:
from pathlib import Path
OUTPUT_ROOT = Path("lesson_44_outputs")
RUN_FULL_SUITE = True
# 本课不需要 API key 或真实模型开关。
# 不要在 Notebook 内核直接调用 os._exit；进程退出实验放在最后的终端命令里。

## 1. 公共约定：身份、数据与工具白名单

本课保留 `read_doc` 与 `calculate`，新增 `create_ticket` 作为**仅本地模拟的写工具**。来源是虚构的：回收站保留 30 天，自动备份保留 14 天。

`canonical()` 让 JSON 键的插入顺序不影响摘要。SHA256 用于检测意外变化，不是加密或来源认证。参数只接受有限的明确字段，不读取任意路径、不执行表达式。

下面同时定义最小的恢复决策函数。后面的独立练习会重新实现它。

In [2]:
from __future__ import annotations
import argparse
import copy
import hashlib
import json
import math
import os
import platform
import re
import sqlite3
import sys
import tempfile
import uuid
from contextlib import contextmanager, ExitStack, closing
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation, localcontext
from pathlib import Path
from typing import Any, Callable

FORMAT = "lesson44-durable-agent-v1"
POLICY_ID = "fixed-evidence-workflow-v1"
TOOL_VERSION = "local-tools-v1"
TASK = "读取回收站与自动备份资料，计算保留天数之差，并仅在本地模拟服务登记一张核查单。"
DOCS = {
    "storage": "虚构教学资料：回收站文件保留30天。",
    "backup": "虚构教学资料：自动备份保留14天。",
}
TOOL_NAMES = {"read_doc", "calculate", "create_ticket"}
CALL_STATES = {"planned", "in_flight", "succeeded", "failed", "unknown"}
RUN_STATES = {"ready", "planning", "waiting", "needs_review", "limit_reached", "final_draft", "aborted"}
SOURCES = {
    "sqlite_python": "https://docs.python.org/3.12/library/sqlite3.html",
    "atomic_commit": "https://www.sqlite.org/atomiccommit.html",
    "idempotency": "https://aws.amazon.com/builders-library/making-retries-safe-with-idempotent-APIs/",
    "retry_limits": "https://docs.aws.amazon.com/wellarchitected/latest/framework/rel_mitigate_interaction_failure_limit_retries.html",
    "agent_persistence": "https://docs.langchain.com/oss/python/langgraph/persistence",
    "unix_lock": "https://docs.python.org/3/library/fcntl.html",
    "windows_lock": "https://docs.python.org/3/library/msvcrt.html",
}

class ContractError(ValueError):
    """不满足输入或持久化约定；禁止静默重置。"""

class VersionConflict(ContractError):
    pass

class BusyRun(ContractError):
    pass

class SimulatedCrash(BaseException):
    """测试专用：不是可重试的普通工具错误。"""


def require(ok: bool, message: str) -> None:
    if not ok:
        raise ContractError(message)


def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def fingerprint(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def strict_json(raw: str) -> Any:
    require(isinstance(raw, str) and len(raw) <= 4_000_000, "JSON为空或过大")
    def pairs(items):
        result = {}
        for key, value in items:
            require(key not in result, f"重复JSON键：{key}")
            result[key] = value
        return result
    def bad_constant(value):
        raise ContractError(f"禁止非有限JSON值：{value}")
    def finite_float(value):
        number = float(value)
        require(math.isfinite(number), "JSON浮点数溢出")
        return number
    try:
        return json.loads(raw, object_pairs_hook=pairs, parse_constant=bad_constant,
                          parse_float=finite_float)
    except (json.JSONDecodeError, RecursionError) as exc:
        raise ContractError("需要完整合法JSON") from exc


def write_json(path: str | Path, data: Any) -> None:
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2,
                                   allow_nan=False) + "\n", encoding="utf-8")


def checked_text(value: Any, name: str, maximum: int = 200) -> str:
    require(isinstance(value, str) and 0 < len(value) <= maximum
            and bool(value.strip()), f"{name}必须是非空短字符串")
    require(not any(ord(c) < 32 for c in value), f"{name}不能含控制字符")
    try:
        value.encode("utf-8")
    except UnicodeError as exc:
        raise ContractError("不接受孤立代理字符") from exc
    return value


def positive_int(value: Any, name: str) -> int:
    require(type(value) is int and value > 0, f"{name}必须是正整数")
    return value


def validate_args(tool: str, arguments: Any) -> dict:
    require(tool in TOOL_NAMES, "工具不在宿主白名单中")
    require(isinstance(arguments, dict), "arguments必须是对象")
    expected = {"read_doc": {"doc_id"},
                "calculate": {"op", "left", "right"},
                "create_ticket": {"title", "body"}}[tool]
    require(set(arguments) == expected, f"{tool}参数集合不正确")
    if tool == "read_doc":
        v = checked_text(arguments["doc_id"], "doc_id", 40)
        require(bool(re.fullmatch(r"[a-z][a-z0-9_]{0,39}", v)), "doc_id不是路径")
    elif tool == "calculate":
        require(arguments["op"] in {"add", "subtract", "multiply", "divide"}, "未知运算")
        for name in ("left", "right"):
            v = checked_text(arguments[name], name, 30)
            require(bool(re.fullmatch(r"-?\d{1,12}(\.\d{1,8})?", v)), "操作数必须是有界十进制字符串")
    else:
        checked_text(arguments["title"], "title", 80)
        checked_text(arguments["body"], "body", 500)
    return copy.deepcopy(arguments)


def operation_key(run_id: str, call_id: str, tool: str, version: str = TOOL_VERSION) -> str:
    """键绑定逻辑意图，不绑定重试次数；参数另存摘要用于拒绝同键不同请求。"""
    for name, value in (("run_id", run_id), ("call_id", call_id), ("tool", tool), ("version", version)):
        checked_text(value, name)
    return fingerprint({"run_id": run_id, "call_id": call_id, "tool": tool, "version": version})


def recovery_action(state: str, replay_kind: str, attempts: int, maximum: int) -> str:
    require(state in CALL_STATES, "未知调用状态")
    require(replay_kind in {"read_only", "idempotent", "unsafe"}, "未知重试契约")
    require(type(attempts) is int and attempts >= 0, "尝试次数非法")
    positive_int(maximum, "maximum")
    if state in {"succeeded", "failed"}:
        return "reuse_result"
    # 非幂等写入的不确定性优先于重试预算；禁止猜测是否已经执行。
    if state in {"in_flight", "unknown"} and replay_kind == "unsafe":
        return "needs_review"
    if attempts >= maximum:
        return "stop_budget"
    if state == "planned":
        return "dispatch"
    return "retry_same_key" if replay_kind == "idempotent" else "repeat_read"


def outcome(call_id: str, tool: str, *, data: dict | None = None,
            error: str | None = None) -> dict:
    return {"call_id": call_id, "name": tool, "ok": error is None,
            "data": data if error is None else None, "error": error}

## 2. 从“消息列表”到“可恢复状态”

只保存 `messages` 无法知道：当前工具是否执行过、结果是否已提交、还剩多少预算、原来的幂等键是什么。

本课保存：会话身份、策略与工具版本、资料摘要、服务身份、配置、消息、调用计划、逐调用状态、尝试次数、总预算、最终草稿。

| 调用状态 | 已知事实 | 恢复时不能臆断什么 |
|---|---|---|
| `planned` | 计划已提交，尚未预留执行尝试 | 不能认定已经产生效果 |
| `in_flight` | 已预留一次分派，结果尚未本地提交 | 可能没发出，也可能已经成功 |
| `succeeded` | 成功结果已持久化 | 不需要重新执行 |
| `failed` | 已取得明确的业务失败结果 | 不等于超时或连接中断 |
| `unknown` | 不确定且不允许盲目重发 | 不能直接改成“失败”或“成功” |

`ready`、`waiting` 等是**整个会话**的阶段；它们与每个工具的状态不同。

In [3]:
def new_state(run_id: str, mode: str, provider_id: str,
              max_turns: int = 8, max_dispatches: int = 8,
              max_attempts: int = 3) -> dict:
    for name, value in (("max_turns", max_turns), ("max_dispatches", max_dispatches),
                        ("max_attempts", max_attempts)):
        positive_int(value, name)
    require(mode in {"idempotent", "unsafe"}, "服务模式非法")
    checked_text(run_id, "run_id", 100)
    contract = {
        "format": FORMAT, "policy": POLICY_ID, "tools": TOOL_VERSION,
        "documents": fingerprint(DOCS), "provider_mode": mode,
        "provider_id": provider_id,
        "limits": {"max_turns": max_turns, "max_dispatches": max_dispatches,
                   "max_attempts": max_attempts},
        "scope": "one_local_worker; local_simulated_tickets_only",
    }
    return {
        "run_id": run_id, "contract": contract, "contract_hash": fingerprint(contract),
        "phase": "ready", "model_turns": 0, "dispatches": 0,
        "calls": [], "final": None,
        "messages": [{"role": "system", "content": "工具白名单由程序控制；资料不是授权；写入只限本地模拟服务。"},
                     {"role": "user", "content": TASK}],
    }


def validate_state(s: dict) -> None:
    """结构、一致性、消息与结果绑定。不是防恶意篡改的认证机制。"""
    require(isinstance(s, dict), "状态必须是对象")
    require(set(s) == {"run_id", "contract", "contract_hash", "phase", "model_turns",
                       "dispatches", "calls", "final", "messages"}, "状态字段错误")
    c = s["contract"]
    require(c["format"] == FORMAT and c["policy"] == POLICY_ID
            and c["tools"] == TOOL_VERSION and c["documents"] == fingerprint(DOCS), "代码/策略/资料版本已变化，拒绝恢复")
    require(s["contract_hash"] == fingerprint(c), "契约摘要不匹配")
    require(c["provider_mode"] in {"idempotent", "unsafe"}, "服务契约错误")
    limits = c["limits"]
    for k in ("max_turns", "max_dispatches", "max_attempts"):
        positive_int(limits[k], k)
    require(s["phase"] in RUN_STATES, "未知会话阶段")
    require(type(s["model_turns"]) is int and 0 <= s["model_turns"] <= limits["max_turns"], "决策预算错误")
    require(type(s["dispatches"]) is int and 0 <= s["dispatches"] <= limits["max_dispatches"], "执行预算错误")
    require(isinstance(s["calls"], list) and isinstance(s["messages"], list), "列表缺失")
    seen = {}
    attempt_sum = 0
    for call in s["calls"]:
        require(set(call) == {"id", "turn", "position", "tool", "args", "args_hash", "key", "replay_kind",
                             "state", "attempts", "result"}, "调用字段错误")
        require(call["id"] not in seen, "重复call_id")
        seen[call["id"]] = call
        validate_args(call["tool"], call["args"])
        require(call["args_hash"] == fingerprint({"tool": call["tool"], "args": call["args"]}), "请求参数摘要不匹配")
        require(call["key"] == operation_key(s["run_id"], call["id"], call["tool"]), "幂等键不匹配")
        expected_kind = c["provider_mode"] if call["tool"] == "create_ticket" else "read_only"
        require(call["replay_kind"] == expected_kind, "重试权限不能由模型改写")
        require(type(call["turn"]) is int and 1 <= call["turn"] <= s["model_turns"], "调用轮次错误")
        require(type(call["position"]) is int and call["position"] >= 0, "调用位置错误")
        require(call["state"] in CALL_STATES, "调用状态错误")
        require(type(call["attempts"]) is int and 0 <= call["attempts"] <= limits["max_attempts"], "调用尝试次数错误")
        require(call["state"] == "planned" or call["attempts"] > 0, "未开始却已有结果")
        attempt_sum += call["attempts"]
        terminal = call["state"] in {"succeeded", "failed"}
        require((call["result"] is not None) == terminal, "调用状态与结果不一致")
        if terminal:
            r = call["result"]
            require(isinstance(r, dict), "工具结果必须是对象")
            require(set(r) == {"call_id", "name", "ok", "data", "error"}, "结果结构错误")
            require(r["call_id"] == call["id"] and r["name"] == call["tool"], "结果与调用不匹配")
            require(type(r["ok"]) is bool and r["ok"] == (call["state"] == "succeeded"), "成功状态错误")
            if r["ok"]:
                require(isinstance(r["data"], dict) and r["error"] is None, "成功结果结构错误")
            else:
                require(r["data"] is None and isinstance(r["error"], str) and bool(r["error"]), "失败结果结构错误")
    require(attempt_sum == s["dispatches"], "总执行计数与逐调用计数不一致")
    pending = set()
    requested = set()
    observed = set()
    for m in s["messages"]:
        role = m.get("role")
        require(role in {"system", "user", "assistant", "tool"}, "未知消息角色")
        if role == "assistant":
            require(not pending, "结果未收齐就出现下一次assistant决策")
            for wire_call in m.get("tool_calls", []):
                cid = wire_call["id"]
                require(cid in seen and cid not in requested, "assistant调用没有对应账本或重复")
                call = seen[cid]
                require(wire_call["function"] == {"name": call["tool"], "arguments": call["args"]}, "历史请求与账本不一致")
                requested.add(cid)
                pending.add(cid)
        elif role == "tool":
            cid = m["tool_call_id"]
            require(cid in pending and cid not in observed, "多余或重复工具观察")
            require(seen[cid]["result"] == m["content"], "消息结果与持久化结果不一致")
            pending.remove(cid)
            observed.add(cid)
    require(requested == set(seen), "持久化计划与assistant历史不一致")
    expected_observed = {c["id"] for c in s["calls"] if c["state"] in {"succeeded", "failed"}}
    require(observed == expected_observed, "完成记录与工具消息必须同时提交")
    if s["phase"] in {"ready", "planning", "final_draft"}:
        require(not pending, "未完成的工具阻塞下一次决策")
    if s["phase"] == "final_draft":
        require(s["final"] is not None and s["messages"][-1]["content"] == s["final"]["text"], "结束状态没有相应草稿")
    else:
        require(s["final"] is None, "尚未结束却已有最终草稿")


def find_call(state: dict, call_id: str) -> dict:
    for call in state["calls"]:
        if call["id"] == call_id:
            return call
    raise ContractError("未知call_id")


def terminal_phase(state: dict) -> str:
    if any(c["state"] == "unknown" for c in state["calls"]):
        return "needs_review"
    if any(c["state"] not in {"succeeded", "failed"} for c in state["calls"]):
        return "waiting"
    return "ready"

In [4]:
example = new_state("demo-run", "idempotent", "simulated-provider")
print("会话阶段：", example["phase"])
print("已使用决策轮次：", example["model_turns"])
print("预算：", example["contract"]["limits"])
print("最初消息角色：", [x["role"] for x in example["messages"]])

会话阶段： ready
已使用决策轮次： 0
预算： {'max_turns': 8, 'max_dispatches': 8, 'max_attempts': 3}
最初消息角色： ['system', 'user']


### 2.1 call ID、重试次数和业务意图不是同一件事

同一个逻辑调用的重试必须复用稳定键；不能每次重试都重新生成 UUID。相同参数却来自两个明确的新意图，也不应被参数哈希全局合并。

本课的键来自 `run_id + call_id + 工具 + 版本`。**参数另存摘要**，服务端对同键不同参数直接报错。调用方应明确表达意图，而不是靠参数相同猜测是否重复。[AWS：安全幂等重试](https://aws.amazon.com/builders-library/making-retries-safe-with-idempotent-APIs/)

In [5]:
key_1 = operation_key("run-A", "call_003_01", "create_ticket")
key_retry = operation_key("run-A", "call_003_01", "create_ticket")
key_new_intent = operation_key("run-A", "call_004_01", "create_ticket")
print("同调用重试同键：", key_1 == key_retry)
print("新调用不同键：", key_1 != key_new_intent)
for kind in ["read_only", "idempotent", "unsafe"]:
    print(kind, "→", recovery_action("in_flight", kind, attempts=1, maximum=3))

同调用重试同键： True
新调用不同键： True
read_only → repeat_read
idempotent → retry_same_key
unsafe → needs_review


## 3. SQLite：先理解一个事务的范围

本课把“完整状态快照＋对应事件”放在**同一个 SQLite 事务**里。要么都提交，要么都没有提交；不允许结果记录与消息历史只更新一半。事务的基本原子性见 [SQLite 官方说明](https://www.sqlite.org/atomiccommit.html)。

使用 `isolation_level=None` 配合显式 `BEGIN IMMEDIATE / COMMIT / ROLLBACK`，避免混用 Python 自动事务与手动事务。所有数据值通过 SQL 参数绑定，而不是拼接成 SQL。[Python sqlite3](https://docs.python.org/3.12/library/sqlite3.html)

工程上用一个 JSON 快照便于阅读；这不是大型服务的高吞吐存储设计。`version` 阻止旧快照覆盖新快照；本地文件锁只允许同一目录一个合作式执行者。

In [6]:
def connect_db(path: Path, *, readonly: bool = False) -> sqlite3.Connection:
    if readonly:
        require(path.is_file(), f"数据库不存在：{path}")
        con = sqlite3.connect(path.resolve().as_uri() + "?mode=ro", uri=True, isolation_level=None)
    else:
        con = sqlite3.connect(str(path), timeout=5, isolation_level=None)
        con.execute("PRAGMA journal_mode=DELETE")
        con.execute("PRAGMA synchronous=FULL")
    con.row_factory = sqlite3.Row
    con.execute("PRAGMA foreign_keys=ON")
    return con


@contextmanager
def transaction(con: sqlite3.Connection):
    require(not con.in_transaction, "不嵌套事务")
    con.execute("BEGIN IMMEDIATE")
    try:
        yield
        con.execute("COMMIT")
    except BaseException:
        if con.in_transaction:
            con.execute("ROLLBACK")
        raise


class RunLock:
    """同一主机、支持锁的本地文件系统上的合作式单执行者；不是分布式租约。"""
    def __init__(self, folder: str | Path):
        self.path = Path(folder) / "runner.lock"
        self.file = None

    def __enter__(self):
        self.file = self.path.open("a+b")
        try:
            if os.name == "nt":
                import msvcrt
                self.file.seek(0, 2)
                if self.file.tell() == 0:
                    self.file.write(b"0")
                    self.file.flush()
                self.file.seek(0)
                msvcrt.locking(self.file.fileno(), msvcrt.LK_NBLCK, 1)
            else:
                import fcntl
                fcntl.flock(self.file.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except (OSError, ImportError) as exc:
            self.file.close()
            self.file = None
            raise BusyRun("同一目录已有执行者，或当前文件系统不支持锁；拒绝恢复") from exc
        return self

    def __exit__(self, *exc):
        if self.file is not None:
            if os.name == "nt":
                import msvcrt
                self.file.seek(0)
                msvcrt.locking(self.file.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                import fcntl
                fcntl.flock(self.file.fileno(), fcntl.LOCK_UN)
            self.file.close()
        # 不删除锁文件；是否存在不是是否仍持有锁的判断依据。


class AgentStore:
    def __init__(self, path: str | Path, *, create: dict | None = None, readonly: bool = False):
        self.path = Path(path)
        require(not (create is not None and self.path.exists()), "拒绝覆盖已有Agent数据库")
        require(create is not None or self.path.is_file(), "找不到Agent数据库，不能当成新会话")
        self.con = connect_db(self.path, readonly=readonly)
        if create is not None:
            validate_state(create)
            self.con.executescript("""
                CREATE TABLE session(id INTEGER PRIMARY KEY CHECK(id=1), version INTEGER NOT NULL,
                                     state_json TEXT NOT NULL, state_hash TEXT NOT NULL);
                CREATE TABLE events(seq INTEGER PRIMARY KEY AUTOINCREMENT, version INTEGER UNIQUE NOT NULL,
                                    kind TEXT NOT NULL, details_json TEXT NOT NULL);
            """)
            with transaction(self.con):
                self.con.execute("INSERT INTO session VALUES(1,0,?,?)", (canonical(create), fingerprint(create)))
                self.con.execute("INSERT INTO events(version,kind,details_json) VALUES(0,'created',?)", (canonical({"run_id": create["run_id"]}),))
        self.load()

    def close(self):
        self.con.close()

    def load(self) -> tuple[int, dict]:
        row = self.con.execute("SELECT version,state_json,state_hash FROM session WHERE id=1").fetchone()
        require(row is not None, "状态表缺失")
        state = strict_json(row["state_json"])
        require(fingerprint(state) == row["state_hash"], "状态内容摘要不匹配")
        validate_state(state)
        return row["version"], state

    def commit(self, expected_version: int, state: dict, kind: str, details: dict | None = None,
               *, fail_before_event: bool = False) -> int:
        validate_state(state)
        with transaction(self.con):
            current = self.con.execute("SELECT version FROM session WHERE id=1").fetchone()[0]
            if current != expected_version:
                raise VersionConflict("快照版本过期，拒绝覆盖较新的结果")
            nxt = current + 1
            self.con.execute("UPDATE session SET version=?,state_json=?,state_hash=? WHERE id=1 AND version=?",
                             (nxt, canonical(state), fingerprint(state), current))
            if fail_before_event:
                raise SimulatedCrash("事务内故障：快照和事件都应回滚")
            self.con.execute("INSERT INTO events(version,kind,details_json) VALUES(?,?,?)",
                             (nxt, kind, canonical(details or {})))
        return nxt

    def mutate(self, change: Callable[[dict], None], kind: str, details: dict | None = None):
        version, state = self.load()
        change(state)
        self.commit(version, state, kind, details)
        return state

    def events(self) -> list[dict]:
        return [{"seq": r["seq"], "version": r["version"], "kind": r["kind"],
                 "details": strict_json(r["details_json"])}
                for r in self.con.execute("SELECT * FROM events ORDER BY seq")]

In [7]:
# 两条 SQL 要么一起完成，要么一起回滚。
con = sqlite3.connect(":memory:", isolation_level=None)
con.execute("CREATE TABLE sample(id INTEGER PRIMARY KEY, value TEXT)")
try:
    with transaction(con):
        con.execute("INSERT INTO sample VALUES(?,?)", (1, "先写入的值"))
        raise RuntimeError("模拟写到一半失败")
except RuntimeError:
    pass
print("失败后的记录数：", con.execute("SELECT count(*) FROM sample").fetchone()[0])
con.close()

失败后的记录数： 0


## 4. 为什么还要独立的 service.sqlite3？

`agent.sqlite3` 是调用方记账；`service.sqlite3` 模拟真正接受工具请求的一侧。

```text
Agent 保存“正在执行”  →  服务提交模拟工单  →  Agent 保存返回结果
```

**两个独立提交不是一个跨服务事务。** 即使本课文件都在一台机器上，也刻意不用 `ATTACH` 把它们合成一个事务，这样才能复现远程调用常见的确认缺口。

幂等服务在自己的事务里同时保存业务效果与幂等回执。同键重试先查回执，返回同一个工单；同键不同参数被拒绝。非幂等服务则每收到一次调用都可能新建一张。

服务端请求审计次数与业务效果数量分开；只读工具的审计日志变化不算用户业务写操作。

In [8]:
class SimulatedService:
    """独立SQLite文件模拟外部边界。不会创建任何真实工单或发送网络请求。"""
    def __init__(self, path: str | Path, *, create: str | None = None, readonly: bool = False):
        self.path = Path(path)
        require(not (create is not None and self.path.exists()), "拒绝覆盖服务数据库")
        require(create is not None or self.path.is_file(), "服务账本丢失，拒绝新建空账本掩盖旧效果")
        self.con = connect_db(self.path, readonly=readonly)
        if create is not None:
            require(create in {"idempotent", "unsafe"}, "未知服务模式")
            self.con.executescript("""
                CREATE TABLE meta(key TEXT PRIMARY KEY, value TEXT NOT NULL);
                CREATE TABLE requests(id INTEGER PRIMARY KEY AUTOINCREMENT, tool TEXT NOT NULL,
                                      operation_key TEXT NOT NULL, request_hash TEXT NOT NULL);
                CREATE TABLE tickets(id INTEGER PRIMARY KEY AUTOINCREMENT, operation_key TEXT NOT NULL,
                                     request_hash TEXT NOT NULL, arguments TEXT NOT NULL);
                CREATE TABLE receipts(operation_key TEXT PRIMARY KEY, request_hash TEXT NOT NULL,
                                      response_json TEXT NOT NULL);
            """)
            with transaction(self.con):
                self.con.executemany("INSERT INTO meta VALUES(?,?)", [
                    ("format", FORMAT), ("mode", create), ("provider_id", uuid.uuid4().hex)])
        meta = dict(self.con.execute("SELECT key,value FROM meta").fetchall())
        require(meta.get("format") == FORMAT and meta.get("mode") in {"unsafe", "idempotent"}, "服务格式/模式错误")
        self.mode, self.provider_id = meta["mode"], meta["provider_id"]

    def close(self):
        self.con.close()

    def dispatch(self, call: dict, *, fail_inside_transaction: bool = False) -> dict:
        """每次到达都记requests；重复请求可能返回同一票据而不新建效果。"""
        tool, args = call["tool"], validate_args(call["tool"], call["args"])
        request_hash = fingerprint({"tool": tool, "args": args})
        require(call["args_hash"] == request_hash, "服务收到的参数摘要不一致")
        key = call["key"]
        with transaction(self.con):
            if tool == "create_ticket" and self.mode == "idempotent":
                old = self.con.execute("SELECT * FROM receipts WHERE operation_key=?", (key,)).fetchone()
                if old is not None:
                    require(old["request_hash"] == request_hash, "同一幂等键不能对应不同参数")
                    self.con.execute("INSERT INTO requests(tool,operation_key,request_hash) VALUES(?,?,?)", (tool, key, request_hash))
                    return strict_json(old["response_json"])
            self.con.execute("INSERT INTO requests(tool,operation_key,request_hash) VALUES(?,?,?)", (tool, key, request_hash))
            if tool == "read_doc":
                doc_id = args["doc_id"]
                result = outcome(call["id"], tool, data={"doc_id": doc_id, "text": DOCS[doc_id],
                                 "revision": fingerprint(DOCS[doc_id])}) if doc_id in DOCS else outcome(call["id"], tool, error="not_found")
            elif tool == "calculate":
                left, right = Decimal(args["left"]), Decimal(args["right"])
                if args["op"] == "divide" and right == 0:
                    result = outcome(call["id"], tool, error="division_by_zero")
                else:
                    with localcontext() as ctx:
                        ctx.prec = 28
                        op = args["op"]
                        number = {"add": lambda: left + right, "subtract": lambda: left - right,
                                  "multiply": lambda: left * right, "divide": lambda: left / right}[op]()
                    result = outcome(call["id"], tool, data={"value": format(number, "f"), "arguments": args,
                                                           "unit_checked": False})
            else:
                cursor = self.con.execute("INSERT INTO tickets(operation_key,request_hash,arguments) VALUES(?,?,?)",
                                          (key, request_hash, canonical(args)))
                result = outcome(call["id"], tool, data={"ticket_id": f"LOCAL-{cursor.lastrowid:04d}",
                                 "arguments": args, "simulated": True})
                if fail_inside_transaction:
                    raise SimulatedCrash("服务事务中断：工单与回执一起回滚")
                if self.mode == "idempotent":
                    self.con.execute("INSERT INTO receipts VALUES(?,?,?)", (key, request_hash, canonical(result)))
        return result

    def lookup_ticket(self, call: dict) -> dict:
        """只查询已提交回执；零条不能证明真实系统里不存在延迟中的请求。"""
        require(call["tool"] == "create_ticket", "本课只对模拟工单提供服务端核对")
        rows = self.con.execute("SELECT * FROM tickets WHERE operation_key=?", (call["key"],)).fetchall()
        require(len(rows) == 1, "没有唯一已提交工单：保持未知，不自动重试")
        row = rows[0]
        require(row["request_hash"] == call["args_hash"] and strict_json(row["arguments"]) == call["args"], "回执参数不匹配")
        return outcome(call["id"], call["tool"], data={"ticket_id": f"LOCAL-{row['id']:04d}",
                       "arguments": strict_json(row["arguments"]), "simulated": True})

    def summary(self) -> dict:
        requests = [dict(r) for r in self.con.execute("SELECT * FROM requests ORDER BY id")]
        tickets = [{"id": r["id"], "operation_key": r["operation_key"], "request_hash": r["request_hash"],
                    "arguments": strict_json(r["arguments"])}
                   for r in self.con.execute("SELECT * FROM tickets ORDER BY id")]
        return {"mode": self.mode, "provider_id": self.provider_id,
                "requests": requests, "request_count": len(requests),
                "ticket_requests": sum(r["tool"] == "create_ticket" for r in requests),
                "ticket_effects": len(tickets), "tickets": tickets}

### 4.1 请求发过两次，不等于效果发生两次

代码会分别向两种模拟服务发送两次相同请求。所有数据库在临时教学目录内，不连接任何外部服务。

In [9]:
def exercise_receipt_demo() -> dict:
    """无网络的幂等性实验：请求两次≠效果两次；同键变参必须拒绝。"""
    results = {}
    with tempfile.TemporaryDirectory(prefix="lesson44_receipts_") as root:
        root = Path(root)
        for mode in ("idempotent", "unsafe"):
            svc = SimulatedService(root / f"{mode}.db", create=mode)
            try:
                args = {"title": "模拟标题", "body": "仅本地效果"}
                call = {"id": "c1", "tool": "create_ticket", "args": args,
                        "args_hash": fingerprint({"tool": "create_ticket", "args": args}),
                        "key": operation_key("r1", "c1", "create_ticket")}
                first, second = svc.dispatch(call), svc.dispatch(call)
                results[mode] = {"first_ticket": first["data"]["ticket_id"],
                                 "second_ticket": second["data"]["ticket_id"],
                                 "requests": svc.summary()["ticket_requests"],
                                 "effects": svc.summary()["ticket_effects"]}
            finally:
                svc.close()
    return results

In [10]:
receipt_demo = exercise_receipt_demo()
for mode, result in receipt_demo.items():
    print(mode, "：", result)

idempotent ： {'first_ticket': 'LOCAL-0001', 'second_ticket': 'LOCAL-0001', 'requests': 2, 'effects': 1}
unsafe ： {'first_ticket': 'LOCAL-0001', 'second_ticket': 'LOCAL-0002', 'requests': 2, 'effects': 2}


## 5. 执行顺序：先落盘计划，再执行工具

本课遵守：

```text
预留决策轮次 → 产生策略输出 → 提交assistant消息、调用计划和稳定键
→ 提交 in_flight 与尝试计数 → 执行工具
→ 同一事务提交调用结果与 tool 消息 → 等本轮所有结果齐全 → 下一轮
```

模型输出已产生但尚未提交时，不能执行工具。恢复时，**已保存的工具计划直接复用**，不再次询问模型“你刚才想做什么”。

故障注入点覆盖提交计划、预留执行、服务提交、本地结果提交等边界。默认异常模拟会正常关闭数据库；真实进程退出实验见最后的命令。

代码串行执行本轮调用，专注恢复机制。上一课的并行执行需要额外的并发状态协调，不能直接无保护地搬进来。

In [11]:
class CrashHook:
    def __init__(self, stage: str | None = None, tool: str | None = None, *, hard: bool = False):
        self.stage, self.tool, self.hard, self.fired = stage, tool, hard, False

    def hit(self, stage: str, tool: str | None = None):
        if not self.fired and self.stage == stage and (self.tool is None or self.tool == tool):
            self.fired = True
            if self.hard:
                print(f"演示进程退出：{stage} / {tool}；退出码75。", flush=True)
                os._exit(75)
            raise SimulatedCrash(f"{stage} / {tool}")


class DurableController:
    """所有修改只应在RunLock内执行；以保存的动作/结果恢复，不重新猜测动作。"""
    def __init__(self, store: AgentStore, service: SimulatedService):
        self.store, self.service = store, service
        _, state = store.load()
        require(state["contract"]["provider_id"] == service.provider_id
                and state["contract"]["provider_mode"] == service.mode, "服务身份或模式变化，拒绝恢复")

    def recover(self):
        _, previous = self.store.load()
        if previous["phase"] in {"final_draft", "aborted", "limit_reached", "needs_review"}:
            return previous
        decisions = []
        def change(s):
            if s["phase"] == "planning":
                # 旧响应尚未提交；没有工具获准执行。重新决策但不退还已消耗预算。
                s["phase"] = "ready"
                decisions.append({"action": "discard_uncommitted_decision", "charged_turn": s["model_turns"]})
            stopped = False
            for call in s["calls"]:
                if call["state"] not in {"in_flight", "unknown"}:
                    continue
                action = recovery_action(call["state"], call["replay_kind"], call["attempts"], s["contract"]["limits"]["max_attempts"])
                decisions.append({"call_id": call["id"], "action": action, "key": call["key"]})
                if action == "needs_review":
                    call["state"] = "unknown"
                elif action == "stop_budget":
                    stopped = True
                else:
                    call["state"] = "planned"
            if any(c["state"] == "unknown" for c in s["calls"]):
                s["phase"] = "needs_review"
            elif stopped:
                s["phase"] = "limit_reached"
            elif s["phase"] != "ready":
                s["phase"] = terminal_phase(s)
        # 没有需恢复状态时不写无意义的新事件。
        if previous["phase"] == "planning" or any(c["state"] in {"in_flight", "unknown"} for c in previous["calls"]):
            return self.store.mutate(change, "recovered", {"decisions": decisions})
        return previous

    def reserve_decision(self) -> bool:
        accepted = [False]
        def change(s):
            require(s["phase"] == "ready", "未收齐结果或会话已停止，不能再次决策")
            if s["model_turns"] >= s["contract"]["limits"]["max_turns"]:
                s["phase"] = "limit_reached"
                return
            s["model_turns"] += 1
            s["phase"] = "planning"
            accepted[0] = True
        self.store.mutate(change, "decision_reserved")
        return accepted[0]

    def commit_action(self, action: dict):
        require(isinstance(action, dict) and action.get("kind") in {"tools", "final"}, "动作格式错误")
        if action["kind"] == "tools":
            require(set(action) == {"kind", "calls"} and isinstance(action["calls"], list)
                    and 1 <= len(action["calls"]) <= 3, "每轮只接受1至3个明确调用")
            for c in action["calls"]:
                require(isinstance(c, dict) and set(c) == {"name", "arguments"}, "调用字段错误")
                validate_args(c["name"], c["arguments"])
        else:
            require(set(action) == {"kind", "text"}, "最终动作字段错误")
            checked_text(action["text"], "text", 1500)
        def change(s):
            require(s["phase"] == "planning", "没有已保留的决策轮次")
            if action["kind"] == "final":
                s["final"] = {"text": action["text"], "task_success": None, "semantic_support_checked": False}
                s["messages"].append({"role": "assistant", "content": action["text"]})
                s["phase"] = "final_draft"
                return
            proposed = action["calls"]
            if s["dispatches"] + len(proposed) > s["contract"]["limits"]["max_dispatches"]:
                s["phase"] = "limit_reached"
                return
            wire = []
            for pos, item in enumerate(proposed):
                cid = f"call_{s['model_turns']:03d}_{pos+1:02d}"
                tool, args = item["name"], copy.deepcopy(item["arguments"])
                call = {"id": cid, "turn": s["model_turns"], "position": pos, "tool": tool,
                        "args": args, "args_hash": fingerprint({"tool": tool, "args": args}),
                        "key": operation_key(s["run_id"], cid, tool),
                        "replay_kind": s["contract"]["provider_mode"] if tool == "create_ticket" else "read_only",
                        "state": "planned", "attempts": 0, "result": None}
                s["calls"].append(call)
                wire.append({"id": cid, "type": "function", "function": {"name": tool, "arguments": args}})
            s["messages"].append({"role": "assistant", "content": None, "tool_calls": wire})
            s["phase"] = "waiting"
        return self.store.mutate(change, "action_committed", {"action": action})

    def start_call(self, call_id: str) -> dict | None:
        selected = []
        def change(s):
            require(s["phase"] == "waiting", "当前阶段不能执行工具")
            call = find_call(s, call_id)
            require(call["state"] == "planned", "只执行计划中的调用，不重做完成调用")
            limits = s["contract"]["limits"]
            if s["dispatches"] >= limits["max_dispatches"] or call["attempts"] >= limits["max_attempts"]:
                s["phase"] = "limit_reached"
                return
            call["attempts"] += 1
            s["dispatches"] += 1
            call["state"] = "in_flight"
            selected.append(copy.deepcopy(call))
        self.store.mutate(change, "dispatch_reserved", {"call_id": call_id})
        return selected[0] if selected else None

    def finish_call(self, call_id: str, result: dict, *, reconciled: bool = False,
                    fail_before_event: bool = False):
        version, s = self.store.load()
        call = find_call(s, call_id)
        if call["state"] in {"succeeded", "failed"}:
            require(call["result"] == result, "不能用不同结果改写已完成调用")
            return s
        require(call["state"] == ("unknown" if reconciled else "in_flight"), "不允许当前状态提交结果")
        require(result["call_id"] == call_id and result["name"] == call["tool"], "结果身份错误")
        call["result"] = copy.deepcopy(result)
        call["state"] = "succeeded" if result["ok"] else "failed"
        s["messages"].append({"role": "tool", "tool_call_id": call_id, "name": call["tool"], "content": copy.deepcopy(result)})
        s["phase"] = terminal_phase(s)
        self.store.commit(version, s, "reconciled" if reconciled else "result_committed",
                          {"call_id": call_id, "result": result,
                           "evidence": "local_service_receipt" if reconciled else "returned_observation"},
                          fail_before_event=fail_before_event)
        return s

    def mark_unknown(self, call_id: str, message: str):
        def change(s):
            call = find_call(s, call_id)
            require(call["state"] == "in_flight", "仅运行中调用可标记未知")
            call["state"] = "unknown"
            s["phase"] = "needs_review"
        self.store.mutate(change, "unexpected_execution_error", {"call_id": call_id, "error": message[:300]})

    def reconcile(self, call_id: str):
        _, state = self.store.load()
        call = find_call(state, call_id)
        require(state["phase"] == "needs_review" and call["state"] == "unknown", "只核对未知结果")
        result = self.service.lookup_ticket(call)
        return self.finish_call(call_id, result, reconciled=True)

    def run(self, policy: Callable[[dict], dict], hook: CrashHook | None = None) -> dict:
        hook = hook or CrashHook()
        self.recover()
        while True:
            _, state = self.store.load()
            if state["phase"] in {"final_draft", "needs_review", "limit_reached", "aborted"}:
                return state
            if state["phase"] == "ready":
                if not self.reserve_decision():
                    continue
                hook.hit("after_decision_reserved")
                _, state = self.store.load()
                action = policy(copy.deepcopy(state))
                hook.hit("after_policy_returned")
                try:
                    state = self.commit_action(action)
                except ContractError as exc:
                    def abort(s):
                        s["phase"] = "aborted"
                    self.store.mutate(abort, "invalid_action", {"error": str(exc)})
                    raise
                tool = action["calls"][0]["name"] if action["kind"] == "tools" else None
                hook.hit("after_plan", tool)
                continue
            require(state["phase"] == "waiting", "不支持的运行状态")
            pending = [c for c in state["calls"] if c["state"] == "planned"]
            require(bool(pending), "没有可执行计划；应先恢复运行中状态")
            call = self.start_call(pending[0]["id"])
            if call is None:
                continue
            hook.hit("after_dispatch_mark", call["tool"])
            try:
                result = self.service.dispatch(call)
            except Exception as exc:
                # 未知异常不能当作“没有效果”：暂停，让操作者查明。
                self.mark_unknown(call["id"], f"{type(exc).__name__}: {exc}")
                raise
            hook.hit("after_service_commit", call["tool"])
            self.finish_call(call["id"], result)
            hook.hit("after_result_commit", call["tool"])

## 6. 固定策略：只隔离控制器，不测模型规划

为了控制变量，本课策略使用程序规则：同轮读取两份资料 → 计算差值 → 创建一张本地模拟核查单 → 输出草稿。

数字从读取结果中解析，最终差值来自计算器。最终消息仍保留 `semantic_support_checked=False` 与 `task_success=None`。完整工作流结束不等于所有陈述都已语义审核。

In [12]:
def demo_policy(state: dict) -> dict:
    """固定规则工作流，专门隔离恢复实验；不代表模型自主规划能力。"""
    calls = state["calls"]
    if any(c["state"] == "failed" for c in calls):
        return {"kind": "final", "text": "有工具明确失败，未宣称任务完成。"}
    if not calls:
        return {"kind": "tools", "calls": [
            {"name": "read_doc", "arguments": {"doc_id": "storage"}},
            {"name": "read_doc", "arguments": {"doc_id": "backup"}},
        ]}
    by_tool = {}
    for call in calls:
        if call["state"] == "succeeded":
            by_tool.setdefault(call["tool"], []).append(call["result"]["data"])
    docs = {x["doc_id"]: x for x in by_tool.get("read_doc", [])}
    require(set(docs) == {"storage", "backup"}, "缺少完整读取结果，不猜测文档内容")
    storage = re.search(r"保留(\d+)天", docs["storage"]["text"])
    backup = re.search(r"保留(\d+)天", docs["backup"]["text"])
    require(storage is not None and backup is not None, "教学资料格式不匹配")
    if "calculate" not in by_tool:
        return {"kind": "tools", "calls": [{"name": "calculate", "arguments": {
            "op": "subtract", "left": storage.group(1), "right": backup.group(1)}}]}
    difference = by_tool["calculate"][0]["value"]
    body = f"回收站{storage.group(1)}天，自动备份{backup.group(1)}天，相差{difference}天。"
    if "create_ticket" not in by_tool:
        return {"kind": "tools", "calls": [{"name": "create_ticket", "arguments": {
            "title": "保留期核查（本地模拟）", "body": body}}]}
    ticket = by_tool["create_ticket"][0]["ticket_id"]
    return {"kind": "final", "text": f"{body}本地模拟核查单：{ticket}；未创建外部工单。"}


def create_session(folder: str | Path, *, mode: str = "idempotent", run_id: str | None = None,
                   max_turns: int = 8, max_dispatches: int = 8, max_attempts: int = 3) -> Path:
    folder = Path(folder).resolve()
    require(not folder.exists(), "目录已存在；新建会话不要复用恢复目录")
    folder.mkdir(parents=True)
    service = SimulatedService(folder / "service.sqlite3", create=mode)
    try:
        state = new_state(run_id or uuid.uuid4().hex, mode, service.provider_id,
                          max_turns, max_dispatches, max_attempts)
        store = AgentStore(folder / "agent.sqlite3", create=state)
        store.close()
    finally:
        service.close()
    return folder


def read_session(folder: str | Path) -> dict:
    """只读状态与服务记录；使用前确认没有正在运行的执行者。"""
    folder = Path(folder).resolve()
    require(folder.is_dir(), "会话目录不存在")
    with ExitStack() as stack:
        store = stack.enter_context(closing(AgentStore(folder / "agent.sqlite3", readonly=True)))
        service = stack.enter_context(closing(SimulatedService(folder / "service.sqlite3", readonly=True)))
        version, state = store.load()
        require(state["contract"]["provider_id"] == service.provider_id, "服务身份已变化")
        return {"version": version, "state": state, "events": store.events(), "service": service.summary()}


def resume_session(folder: str | Path, *, hook: CrashHook | None = None) -> dict:
    folder = Path(folder).resolve()
    require(folder.is_dir(), "会话目录不存在")
    with RunLock(folder):
        with ExitStack() as stack:
            store = stack.enter_context(closing(AgentStore(folder / "agent.sqlite3")))
            service = stack.enter_context(closing(SimulatedService(folder / "service.sqlite3")))
            controller = DurableController(store, service)
            controller.run(demo_policy, hook)
        report = read_session(folder)
        write_json(folder / "latest_report.json", report)
        return report


def reconcile_session(folder: str | Path, call_id: str) -> dict:
    """操作者显式核对本地模拟回执；不执行新的写操作，也不接受任意手填成功。"""
    folder = Path(folder).resolve()
    require(folder.is_dir(), "会话目录不存在")
    with RunLock(folder):
        with ExitStack() as stack:
            store = stack.enter_context(closing(AgentStore(folder / "agent.sqlite3")))
            service = stack.enter_context(closing(SimulatedService(folder / "service.sqlite3")))
            DurableController(store, service).reconcile(call_id)
        report = read_session(folder)
        write_json(folder / "latest_report.json", report)
        return report


def short_report(report: dict) -> dict:
    state, service = report["state"], report["service"]
    return {"phase": state["phase"], "decision_turns": state["model_turns"],
            "dispatch_reservations": state["dispatches"],
            "service_requests": service["request_count"],
            "ticket_requests": service["ticket_requests"], "ticket_effects": service["ticket_effects"],
            "call_states": {c["id"]: c["state"] for c in state["calls"]}, "final": state["final"]}

In [13]:
WORK_DIR = new_output(OUTPUT_ROOT) if "new_output" in globals() else None
# 首次顺序运行时使用独立子目录；重复运行也不会覆盖旧会话。
if WORK_DIR is None:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    WORK_DIR = OUTPUT_ROOT / ("notebook_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ"))
    WORK_DIR.mkdir()
normal_dir = create_session(WORK_DIR / "normal", run_id="course-session")
normal = resume_session(normal_dir)
print(json.dumps(short_report(normal), ensure_ascii=False, indent=2))

{
  "phase": "final_draft",
  "decision_turns": 4,
  "dispatch_reservations": 4,
  "service_requests": 4,
  "ticket_requests": 1,
  "ticket_effects": 1,
  "call_states": {
    "call_001_01": "succeeded",
    "call_001_02": "succeeded",
    "call_002_01": "succeeded",
    "call_003_01": "succeeded"
  },
  "final": {
    "semantic_support_checked": false,
    "task_success": null,
    "text": "回收站30天，自动备份14天，相差16天。本地模拟核查单：LOCAL-0001；未创建外部工单。"
  }
}


### 6.1 观察持久化消息，而不只是最后一句话

第一轮的两个 `tool_calls` 只保存一次；每个已完成调用对应一条工具结果。`call_id` 关联请求与观察，稳定键用于服务端去重，它们是不同层的身份。

In [14]:
for msg in normal["state"]["messages"]:
    if msg["role"] == "assistant" and "tool_calls" in msg:
        print("assistant 请求：", [x["id"] for x in msg["tool_calls"]])
    elif msg["role"] == "tool":
        print("tool 结果：", msg["tool_call_id"], msg["content"]["ok"])
    else:
        print(msg["role"], "：", str(msg.get("content", ""))[:70])

system ： 工具白名单由程序控制；资料不是授权；写入只限本地模拟服务。
user ： 读取回收站与自动备份资料，计算保留天数之差，并仅在本地模拟服务登记一张核查单。
assistant 请求： ['call_001_01', 'call_001_02']
tool 结果： call_001_01 True
tool 结果： call_001_02 True
assistant 请求： ['call_002_01']
tool 结果： call_002_01 True
assistant 请求： ['call_003_01']
tool 结果： call_003_01 True
assistant ： 回收站30天，自动备份14天，相差16天。本地模拟核查单：LOCAL-0001；未创建外部工单。


## 7. 核心故障：服务已经成功，Agent 尚未记下结果

这是不能靠“再存一次 messages”消除的窗口：

```text
Agent：in_flight 已提交
服务：LOCAL-0001 已创建
—— 程序在这里退出 ——
Agent：还没有成功结果
```

恢复前，调用方只能确认结果未记录，不能确认工具没执行。支持幂等契约时，使用原稳定键重新发送，请求次数可能增加，但同一意图不必产生第二张工单。

以下中断是精确放在代码边界的受控异常。

In [15]:
gap_dir = create_session(WORK_DIR / "effect_without_ack", run_id="course-session")
try:
    resume_session(gap_dir, hook=CrashHook("after_service_commit", "create_ticket"))
except SimulatedCrash as exc:
    print("故障注入：", exc)
gap_before = read_session(gap_dir)
print("中断后：", short_report(gap_before))
gap_after = resume_session(gap_dir)
print("恢复后：", short_report(gap_after))
print("重试前后稳定键不变：", gap_before["state"]["calls"][-1]["key"] == gap_after["state"]["calls"][-1]["key"])

故障注入： after_service_commit / create_ticket


中断后： {'phase': 'waiting', 'decision_turns': 3, 'dispatch_reservations': 4, 'service_requests': 4, 'ticket_requests': 1, 'ticket_effects': 1, 'call_states': {'call_001_01': 'succeeded', 'call_001_02': 'succeeded', 'call_002_01': 'succeeded', 'call_003_01': 'in_flight'}, 'final': None}


恢复后： {'phase': 'final_draft', 'decision_turns': 4, 'dispatch_reservations': 5, 'service_requests': 5, 'ticket_requests': 2, 'ticket_effects': 1, 'call_states': {'call_001_01': 'succeeded', 'call_001_02': 'succeeded', 'call_002_01': 'succeeded', 'call_003_01': 'succeeded'}, 'final': {'semantic_support_checked': False, 'task_success': None, 'text': '回收站30天，自动备份14天，相差16天。本地模拟核查单：LOCAL-0001；未创建外部工单。'}}
重试前后稳定键不变： True


### 7.1 已保存的结果应直接复用

如果中断发生在本地结果提交之后，恢复不能重新执行已经完成的调用。两份文档只读了一份就退出时，恢复也只处理剩下的一份；没有收齐结果前不允许下一次决策。

In [16]:
def crash_case(folder: Path, stage: str | None, tool: str | None = "create_ticket",
               mode: str = "idempotent", **limits) -> dict:
    create_session(folder, mode=mode, run_id="course-session", **limits)
    interrupted = False
    try:
        resume_session(folder, hook=CrashHook(stage, tool))
    except SimulatedCrash:
        interrupted = True
    before = read_session(folder)
    after = resume_session(folder)
    result = {"interrupted": interrupted, "stage": stage, "mode": mode,
              "before": short_report(before), "after": short_report(after)}
    if mode == "unsafe" and after["state"]["phase"] == "needs_review":
        cid = next(c["id"] for c in after["state"]["calls"] if c["state"] == "unknown")
        try:
            reconcile_session(folder, cid)
            final = resume_session(folder)
            result["after_reconciliation"] = short_report(final)
        except ContractError as exc:
            result["reconciliation_refused"] = str(exc)
    write_json(folder / "case.json", result)
    return result


def scenario_suite(root: str | Path) -> dict:
    root = Path(root)
    specs = [
        ("uninterrupted", None, None, "idempotent", {}),
        ("plan_saved", "after_plan", "create_ticket", "idempotent", {}),
        ("marked_not_sent", "after_dispatch_mark", "create_ticket", "idempotent", {}),
        ("effect_without_ack", "after_service_commit", "create_ticket", "idempotent", {}),
        ("result_saved", "after_result_commit", "create_ticket", "idempotent", {}),
        ("partial_read_batch", "after_result_commit", "read_doc", "idempotent", {}),
        ("decision_not_saved", "after_policy_returned", None, "idempotent", {}),
        ("unsafe_effect_unknown", "after_service_commit", "create_ticket", "unsafe", {}),
        ("unsafe_mark_only", "after_dispatch_mark", "create_ticket", "unsafe", {}),
        ("retry_budget", "after_service_commit", "create_ticket", "idempotent", {"max_attempts": 1}),
        ("total_budget", "after_service_commit", "create_ticket", "idempotent", {"max_dispatches": 4}),
    ]
    return {name: crash_case(root / name, stage, tool, mode, **limits)
            for name, stage, tool, mode, limits in specs}


def new_output(root: str | Path) -> Path:
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    folder = root / f"offline_{stamp}_{uuid.uuid4().hex[:6]}"
    folder.mkdir()
    return folder


def run_offline(root: str | Path = "lesson_44_outputs", checks: dict | None = None) -> Path:
    folder = new_output(root)
    summary = scenario_suite(folder)
    write_json(folder / "summary.json", summary)
    write_json(folder / "idempotency.json", exercise_receipt_demo())
    if checks is not None:
        write_json(folder / "checks.json", checks)
    write_json(folder / "manifest.json", {
        "format": FORMAT, "mode": "standard_library_local_simulation", "status": "completed",
        "python": platform.python_version(), "sqlite": sqlite3.sqlite_version,
        "platform": platform.platform(), "sources": SOURCES,
        "limitations": ["没有模型调用或真实网络工具", "固定规则策略不等于模型规划",
                        "单机合作式单执行者，不是分布式租约", "受控中断不等于断电测试",
                        "无通用exactly-once保证", "无加密、无认证、无生产写操作授权",
                        "恢复不证明最终回答语义正确"]})
    return folder

In [17]:
partial = crash_case(WORK_DIR / "partial_batch", "after_result_commit", "read_doc")
print("部分完成时：", partial["before"]["call_states"])
print("恢复后总请求数：", partial["after"]["service_requests"])
saved = crash_case(WORK_DIR / "saved_result", "after_result_commit", "create_ticket")
print("已存结果后恢复的工单请求数：", saved["after"]["ticket_requests"])
print("已存结果后恢复的工单效果数：", saved["after"]["ticket_effects"])

部分完成时： {'call_001_01': 'succeeded', 'call_001_02': 'planned'}
恢复后总请求数： 4
已存结果后恢复的工单请求数： 1
已存结果后恢复的工单效果数： 1


## 8. 非幂等写入：把“不知道”保留下来

使用 `unsafe` 模拟服务时，同键再次发送仍可能创建第二张工单。恢复看到 `in_flight` 必须先转成 `unknown / needs_review`，**不能把网络中断当成业务失败，然后自动重发**。

注意两个可能的真实世界：

| 同一个本地状态 | 服务侧真实情况 |
|---|---|
| `in_flight` | 还没来得及发送，效果为零 |
| `in_flight` | 服务已提交，但确认没保存，效果为一 |

仅靠本地状态无法区分。下面两次实验在不同边界中断，展示这个信息缺口。

In [18]:
unsafe_sent = crash_case(WORK_DIR / "unsafe_sent", "after_service_commit", "create_ticket", "unsafe")
unsafe_unsent = crash_case(WORK_DIR / "unsafe_unsent", "after_dispatch_mark", "create_ticket", "unsafe")
print("服务已提交但缺确认：", unsafe_sent["after"])
print("尚未发送也不确定：", unsafe_unsent["after"])
print("存在唯一回执时，核对后：", unsafe_sent["after_reconciliation"]["phase"])
print("没有唯一回执时：", unsafe_unsent["reconciliation_refused"])

服务已提交但缺确认： {'phase': 'needs_review', 'decision_turns': 3, 'dispatch_reservations': 4, 'service_requests': 4, 'ticket_requests': 1, 'ticket_effects': 1, 'call_states': {'call_001_01': 'succeeded', 'call_001_02': 'succeeded', 'call_002_01': 'succeeded', 'call_003_01': 'unknown'}, 'final': None}
尚未发送也不确定： {'phase': 'needs_review', 'decision_turns': 3, 'dispatch_reservations': 4, 'service_requests': 3, 'ticket_requests': 0, 'ticket_effects': 0, 'call_states': {'call_001_01': 'succeeded', 'call_001_02': 'succeeded', 'call_002_01': 'succeeded', 'call_003_01': 'unknown'}, 'final': None}
存在唯一回执时，核对后： final_draft
没有唯一回执时： 没有唯一已提交工单：保持未知，不自动重试


### 8.1 核对回执，不是手工填写“成功”

本课的 `--reconcile` 是操作者显式发起的只读查询。它要求模拟服务里存在**唯一且参数摘要一致**的已提交工单，再补写本地观察，不创建新工单。

真实服务可能通过操作号查询状态，但需要供应方契约。“暂时查不到”不总能排除延迟中的请求。本课没有唯一回执时保持未知，不提供直接强制标成功或无条件重试的按钮。

本地回执核对没有用户登录或角色认证，因此它是教学流程，不是生产审批系统。

In [19]:
print("核对前工单请求：", unsafe_sent["after"]["ticket_requests"])
print("核对后工单请求：", unsafe_sent["after_reconciliation"]["ticket_requests"])
print("核对后工单效果：", unsafe_sent["after_reconciliation"]["ticket_effects"])
review_report = read_session(WORK_DIR / "unsafe_sent")
print("核对事件：", [e["kind"] for e in review_report["events"] if e["kind"] == "reconciled"])

核对前工单请求： 1
核对后工单请求： 1
核对后工单效果： 1
核对事件： ['reconciled']


## 9. 状态和事件原子提交；服务效果和回执也原子提交

下面故意在 Agent 快照更新后、事件写入前抛错。事务回滚后应看到原版本、原状态、原事件数量。

这保证的是**本地数据库事务的范围**。它不能撤回此前已经在另一系统完成的动作。SQLite 的硬件与文件系统假设仍然重要；课堂故障不是全套断电与磁盘损坏测试。[SQLite 原子提交与限制](https://www.sqlite.org/atomiccommit.html)

In [20]:
atomic_dir = create_session(WORK_DIR / "atomic_snapshot")
store = AgentStore(atomic_dir / "agent.sqlite3")
try:
    old_version, old_state = store.load()
    candidate = copy.deepcopy(old_state)
    candidate["phase"] = "planning"
    candidate["model_turns"] = 1
    try:
        store.commit(old_version, candidate, "demo", fail_before_event=True)
    except SimulatedCrash:
        pass
    print("快照完全回滚：", store.load() == (old_version, old_state))
    print("事件数量：", len(store.events()))
finally:
    store.close()

快照完全回滚： True
事件数量： 1


## 10. 恢复不能让预算归零

重试也消耗资源。服务的重复请求仍然会传输、解析、查询回执；幂等不代表免费。生产服务还需要超时、退避、抖动与最大重试限制，不能对所有错误无限重试。[AWS 重试限制](https://docs.aws.amazon.com/wellarchitected/latest/framework/rel_mitigate_interaction_failure_limit_retries.html)

本课没有网络，重试由明确恢复操作触发，不模拟长时间等待或退避。保存以下计数：决策轮次、分派预留数、每个调用的尝试数。发送前已预留却崩溃也消耗次数，所以**分派预留数不一定等于服务实际收到次数**。

下面限制工单最多尝试一次。服务已创建它，但重试预算耗尽，恢复不会重发。状态显示 `limit_reached`，不能据此宣称业务没有发生。

In [21]:
limited = crash_case(WORK_DIR / "retry_limit", "after_service_commit", "create_ticket",
                     "idempotent", max_attempts=1)
print(json.dumps(limited["after"], ensure_ascii=False, indent=2))
# 恢复未知结果与重置预算是两回事。
print("幂等确认缺口示例，恢复后预留尝试数：", gap_after["state"]["dispatches"])

{
  "phase": "limit_reached",
  "decision_turns": 3,
  "dispatch_reservations": 4,
  "service_requests": 4,
  "ticket_requests": 1,
  "ticket_effects": 1,
  "call_states": {
    "call_001_01": "succeeded",
    "call_001_02": "succeeded",
    "call_002_01": "succeeded",
    "call_003_01": "in_flight"
  },
  "final": null
}
幂等确认缺口示例，恢复后预留尝试数： 5


### 10.1 模型已经返回，但计划还未保存

这与“工具完成但结果未保存”不同。尚未提交的计划没有获得工具执行权，恢复可以重新决策，但已经预留的决策预算不能退还。

本课策略是确定性的；真实语言模型重新生成可能改变行动，即使设置相同随机种子也不能承诺跨设备完全相同。

已经提交的计划不重新生成。只保存工作流状态，也不等于恢复模型内部执行位置或 KV cache。

In [22]:
lost_decision = crash_case(WORK_DIR / "lost_decision", "after_policy_returned", None)
print("连续运行决策轮次：", short_report(normal)["decision_turns"])
print("重新决策后的轮次：", lost_decision["after"]["decision_turns"])
print("本地工单效果：", lost_decision["after"]["ticket_effects"])

连续运行决策轮次： 4
重新决策后的轮次： 5
本地工单效果： 1


## 11. 单执行者与过期快照

恢复者首先获取本机文件锁。旧进程仍持锁时，新进程拒绝接管，不能因为“已经等了几秒”就认定旧工具停止。

锁文件可以一直存在，操作系统持有的锁才表示忙碌。进程退出后锁会释放；不要手动删除锁文件来抢占。POSIX 用 `flock`，Windows 分支使用字节范围锁。[Unix 文档](https://docs.python.org/3/library/fcntl.html) · [Windows 文档](https://docs.python.org/3/library/msvcrt.html)

这不覆盖分布式多主机、网络文件系统、恶意绕过本程序的执行者或远端仍在运行的工具。真正多执行者需要额外的租约、隔离与 fencing 设计，本课没有实现。

In [23]:
with RunLock(normal_dir):
    try:
        resume_session(normal_dir)
    except BusyRun as exc:
        print("第二执行者被拒绝：", exc)

store = AgentStore(atomic_dir / "agent.sqlite3")
try:
    version, state = store.load()
    store.commit(version, state, "heartbeat_demo")
    try:
        store.commit(version, state, "stale_overwrite")
    except VersionConflict as exc:
        print("过期快照被拒绝：", exc)
finally:
    store.close()

第二执行者被拒绝： 同一目录已有执行者，或当前文件系统不支持锁；拒绝恢复
过期快照被拒绝： 快照版本过期，拒绝覆盖较新的结果


## 12. 检查点、日志回看、恢复和重新运行

| 操作 | 是否执行工具？ | 是否继续同一意图？ |
|---|---|---|
| `--show` | 否，只读取已有记录 | 不推进任务 |
| `--resume` | 只处理未完成且允许继续的调用 | 是，沿用稳定键与预算 |
| `--start 新目录` | 新会话会重新执行 | 是新的意图，可能创建新效果 |
| 阅读 `events` | 否 | 只帮助理解历史 |

**不要为了恢复去创建新目录和新 `run_id`。** 新会话的幂等键不同，服务不能知道你其实想恢复旧意图。

已完成会话重复恢复不调用策略和工具。下面直接核对状态、事件与服务账本。

In [24]:
before_completed_resume = read_session(normal_dir)
after_completed_resume = resume_session(normal_dir)
print("已完成会话再次恢复，数据库语义状态不变：", before_completed_resume == after_completed_resume)
print("最后草稿：", after_completed_resume["state"]["final"])

已完成会话再次恢复，数据库语义状态不变： True
最后草稿： {'semantic_support_checked': False, 'task_success': None, 'text': '回收站30天，自动备份14天，相差16天。本地模拟核查单：LOCAL-0001；未创建外部工单。'}


## 13. 一次运行完整故障矩阵

本课准备 11 个独立场景。表中 `ticket_requests` 只统计模拟工单接口收到的请求，`ticket_effects` 统计真正创建的模拟工单；不混入两个读取和一次计算。

`unsafe_effect_unknown` 的 `after` 是暂停状态；`after_reconciliation` 才是显式核对后的结果。非幂等服务“发送前中断”和“成功但未记录”都先暂停，这是有意保守的恢复规则。

In [25]:
if RUN_FULL_SUITE:
    FULL_RUN = run_offline(OUTPUT_ROOT)
    full_summary = strict_json((FULL_RUN / "summary.json").read_text(encoding="utf-8"))
    for name, row in full_summary.items():
        r = row["after"]
        print(f"{name:24s} {r['phase']:14s} 工单请求={r['ticket_requests']} 效果={r['ticket_effects']}")
    print("完整报告目录：", FULL_RUN.resolve())

uninterrupted            final_draft    工单请求=1 效果=1
plan_saved               final_draft    工单请求=1 效果=1
marked_not_sent          final_draft    工单请求=1 效果=1
effect_without_ack       final_draft    工单请求=2 效果=1
result_saved             final_draft    工单请求=1 效果=1
partial_read_batch       final_draft    工单请求=1 效果=1
decision_not_saved       final_draft    工单请求=1 效果=1
unsafe_effect_unknown    needs_review   工单请求=1 效果=1
unsafe_mark_only         needs_review   工单请求=0 效果=0
retry_budget             limit_reached  工单请求=1 效果=1
total_budget             limit_reached  工单请求=1 效果=1
完整报告目录： /mnt/data/_lesson44_validation/notebook_run/lesson_44_outputs/offline_20261102T083753_181267Z_558ba7


### 文件在哪里？什么是恢复依据？

每个案例目录包含：

| 文件 | 用途 |
|---|---|
| `agent.sqlite3` | 权威的状态、消息、调用与事件 |
| `service.sqlite3` | 独立模拟服务的请求记录、工单与幂等回执 |
| `runner.lock` | 本机单执行者锁；文件存在不表示正在运行 |
| `case.json` | 故障前后摘要 |
| `latest_report.json` | 一次导出的可读报告，不是恢复依据 |

报告可能比数据库旧，例如进程恰好在导出前退出。恢复读取数据库，不读取旧 JSON 猜测状态。

**不要单独删除服务账本，再用空服务继续恢复；不要复制一半数据库伪装成完整会话。** 若未来改用 WAL，也不能只复制一个仍在写入的 `.db` 文件而忽略相关状态。

所有内容明文保存在磁盘，摘要不是签名。请不要放密码、访问令牌或私人资料；本课不实现加密、认证、审计防篡改和数据清理策略。

## 14. 练习检查器

下面只定义检查函数，不会替你完成练习。A/B 检查稳定身份与恢复选择；C 使用内存 SQLite 模拟事务中途失败。

In [26]:
def exercise_checks(key_fn=None, recovery_fn=None, atomic_fn=None) -> dict:
    counts = {}
    if key_fn is not None:
        a = key_fn("r1", "c1", "read_doc")
        require(a == operation_key("r1", "c1", "read_doc"), "A：稳定键格式不匹配")
        require(a == key_fn("r1", "c1", "read_doc"), "A：重试键必须稳定")
        require(a != key_fn("r2", "c1", "read_doc"), "A：不能混用会话")
        require(a != key_fn("r1", "c2", "read_doc"), "A：不同意图不共用键")
        require(a != key_fn("r1", "c1", "calculate"), "A：工具命名空间不能混用")
        require(a != key_fn("r1", "c1", "read_doc", "v2"), "A：版本须进入身份")
        counts["A"] = 6
    if recovery_fn is not None:
        cases = [("succeeded", "unsafe", 1, 1, "reuse_result"),
                 ("failed", "read_only", 1, 2, "reuse_result"),
                 ("planned", "unsafe", 0, 2, "dispatch"),
                 ("in_flight", "read_only", 1, 2, "repeat_read"),
                 ("in_flight", "idempotent", 1, 2, "retry_same_key"),
                 ("unknown", "unsafe", 1, 1, "needs_review"),
                 ("in_flight", "idempotent", 2, 2, "stop_budget"),
                 ("planned", "read_only", 2, 2, "stop_budget")]
        for *args, expected in cases:
            require(recovery_fn(*args) == expected, f"B：{args}处理错误")
        counts["B"] = len(cases)
    if atomic_fn is not None:
        con = sqlite3.connect(":memory:", isolation_level=None)
        con.executescript("CREATE TABLE calls(id TEXT PRIMARY KEY,state TEXT,result TEXT);"
                          "CREATE TABLE observations(call_id TEXT PRIMARY KEY,result TEXT);"
                          "INSERT INTO calls VALUES('c1','in_flight',NULL);")
        try:
            try:
                atomic_fn(con, "c1", {"value": "16"}, True)
            except RuntimeError:
                pass
            else:
                raise ContractError("C：故障必须可观察")
            require(tuple(con.execute("SELECT state,result FROM calls").fetchone()) == ("in_flight", None), "C：调用更新未回滚")
            require(con.execute("SELECT count(*) FROM observations").fetchone()[0] == 0, "C：失败时不能留下半条消息")
            atomic_fn(con, "c1", {"value": "16"}, False)
            require(con.execute("SELECT state FROM calls").fetchone()[0] == "succeeded", "C：未提交完成状态")
            require(strict_json(con.execute("SELECT result FROM observations").fetchone()[0]) == {"value": "16"}, "C：结果未提交")
            require(not con.in_transaction, "C：没有关闭事务")
        finally:
            con.close()
        counts["C"] = 6
    return counts

### 练习 A：稳定操作键

实现 `operation_key_ex(run_id, call_id, tool, version)`：使用规范 JSON 与 SHA256。

同一个逻辑调用的重试应产生相同键；不同会话、工具版本或新的调用 ID 应产生不同键。**不要把当前时间、尝试次数或新的随机 UUID 加进来**。参数内容另存摘要供冲突检查。

完成后开启本单元开关。

In [27]:
def operation_key_ex(run_id, call_id, tool, version=TOOL_VERSION):
    raise NotImplementedError("由你实现稳定键")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    print(exercise_checks(key_fn=operation_key_ex))

### 练习 B：恢复动作判定

实现 `recovery_action_ex(state, replay_kind, attempts, maximum)`，返回约定字符串：

- 已完成或明确失败：`reuse_result`。
- 尚未执行且有预算：`dispatch`。
- 不确定只读调用：`repeat_read`；不确定幂等写入：`retry_same_key`。
- 不确定非幂等写入：`needs_review`，即使预算也耗尽，先保留未知属性。
- 其他允许重试的情况若次数已用尽：`stop_budget`。

该判定假设原执行者已退出、恢复者获得锁，不能拿它绕过锁抢占仍在运行的工具。

In [28]:
def recovery_action_ex(state, replay_kind, attempts, maximum):
    raise NotImplementedError("由你实现恢复策略")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    print(exercise_checks(recovery_fn=recovery_action_ex))

### 练习 C：结果与消息一起提交

检查器会提供两张简表：

```sql
calls(id TEXT PRIMARY KEY, state TEXT, result TEXT)
observations(call_id TEXT PRIMARY KEY, result TEXT)
```

实现 `atomic_result_ex(con, call_id, result, fail_between=False)`：

开始事务，确认原状态为 `in_flight`，将调用改成 `succeeded`，插入对应观察，提交。`fail_between=True` 时，在更新调用后、插入观察前抛出 `RuntimeError`。

发生异常时回滚并继续抛出。检查器将验证失败时两表均不变，成功时两表一起更新。数据值使用 SQL 参数，不拼接字符串。

In [29]:
def atomic_result_ex(con, call_id, result, fail_between=False):
    raise NotImplementedError("由你实现原子提交")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    print(exercise_checks(atomic_fn=atomic_result_ex))

## 15. 参考答案

先完成练习，再阅读下面的实现。参考函数名称不同，不会覆盖你的代码。

In [30]:
def operation_key_reference(run_id, call_id, tool, version=TOOL_VERSION):
    for name, value in (("run_id", run_id), ("call_id", call_id), ("tool", tool), ("version", version)):
        checked_text(value, name)
    data = {"run_id": run_id, "call_id": call_id, "tool": tool, "version": version}
    return hashlib.sha256(canonical(data).encode("utf-8")).hexdigest()


def recovery_action_reference(state, replay_kind, attempts, maximum):
    require(state in CALL_STATES and replay_kind in {"read_only", "idempotent", "unsafe"}, "状态/契约错误")
    require(type(attempts) is int and attempts >= 0, "尝试次数非法")
    positive_int(maximum, "maximum")
    if state in ("succeeded", "failed"):
        return "reuse_result"
    if state in ("in_flight", "unknown") and replay_kind == "unsafe":
        return "needs_review"
    if attempts >= maximum:
        return "stop_budget"
    if state == "planned":
        return "dispatch"
    return "repeat_read" if replay_kind == "read_only" else "retry_same_key"


def atomic_result_reference(con, call_id, result, fail_between=False):
    """练习用简表：调用完成与消息落盘同一事务，禁止状态只更新一半。"""
    require(not con.in_transaction, "需要独立事务")
    con.execute("BEGIN IMMEDIATE")
    try:
        row = con.execute("SELECT state FROM calls WHERE id=?", (call_id,)).fetchone()
        require(row is not None and row[0] == "in_flight", "调用不是运行中")
        encoded = canonical(result)
        con.execute("UPDATE calls SET state='succeeded',result=? WHERE id=?", (encoded, call_id))
        if fail_between:
            raise RuntimeError("模拟在两条SQL之间失败")
        con.execute("INSERT INTO observations(call_id,result) VALUES(?,?)", (call_id, encoded))
        con.execute("COMMIT")
    except BaseException:
        if con.in_transaction:
            con.execute("ROLLBACK")
        raise

In [31]:
print(exercise_checks(operation_key_reference, recovery_action_reference, atomic_result_reference))

{'A': 6, 'B': 8, 'C': 6}


## 16. 自动检查与边界

检查覆盖：输入和状态约定、稳定键、11种恢复场景、明确失败、同键不同参数、服务端事务回滚、快照原子提交、重复结果、消息完整性、预算、过期快照、本地互斥、服务账本丢失与替换。

`Checks` 使用显式异常，不依赖可能被 `python -O` 移除的 `assert`。

检查通过意味着这些有限案例被验证，不意味着任意 Agent、任意远程 API 都能恰好执行一次。

**本次生成验证记录：** Linux 环境中，64 个单元内的 33 个代码单元已从全新内核执行成功，88 项固定检查通过；三组参考答案分别在新进程验证。脚本与 Notebook 的 `summary.json`、`idempotency.json`、`checks.json` 内容一致。

另外在独立进程中，使用 `os._exit(75)` 在服务提交后退出，再启动新进程恢复：服务收到两次工单请求，只创建一张工单；部分读取批次恢复时不重做已完成读取。非幂等模式在人工发起回执查询前一直停留在 `needs_review`。还检查了未提交 SQLite 事务在进程退出后回滚。**这不是断电或真实网络测试。**

五种故意注入的错误均被原有检查发现：未知非幂等操作盲目重试、同键不同参数被放行、异常时提交而不是回滚、每次重试生成新键、完成调用却遗漏工具消息。Windows 文件锁分支未实测。

In [32]:
class Checks:
    def __init__(self):
        self.names = []
    def ok(self, value, name):
        if not value:
            raise AssertionError(name)
        self.names.append(name)
    def raises(self, exc, fn, name):
        try:
            fn()
        except exc:
            self.names.append(name)
        else:
            raise AssertionError(name + "：未抛出预期异常")


def run_checks() -> dict:
    c = Checks()
    c.ok(canonical({"b": 2, "a": 1}) == canonical({"a": 1, "b": 2}), "规范JSON不受键插入顺序影响")
    c.raises(ContractError, lambda: strict_json('{"x":1,"x":2}'), "拒绝重复JSON键")
    c.raises(ContractError, lambda: strict_json('{"x":NaN}'), "拒绝NaN")
    c.raises(ContractError, lambda: strict_json('{"x":1e999}'), "拒绝浮点溢出")
    c.raises(ContractError, lambda: validate_args("shell", {}), "未授权工具不进入执行器")
    c.raises(ContractError, lambda: validate_args("read_doc", {"doc_id": "../secret"}), "文档ID不能是路径")
    c.raises(ContractError, lambda: validate_args("calculate", {"op": "add", "left": 1, "right": "2"}), "数字字符串类型明确")
    c.raises(ContractError, lambda: validate_args("create_ticket", {"title": "x", "body": "y", "approved": True}), "模型不能添加授权字段")
    c.ok(operation_key("r1", "c1", "read_doc") == operation_key("r1", "c1", "read_doc"), "重试不改变稳定键")
    c.ok(operation_key("r1", "c1", "read_doc") != operation_key("r2", "c1", "read_doc"), "不同会话隔离")
    c.ok(operation_key("r1", "c1", "read_doc") != operation_key("r1", "c2", "read_doc"), "同参数不同意图不混用")
    c.ok(recovery_action("in_flight", "unsafe", 1, 3) == "needs_review", "不安全写入结果未知不重发")
    c.ok(recovery_action("in_flight", "idempotent", 1, 3) == "retry_same_key", "幂等写入复用键重试")
    c.ok(recovery_action("in_flight", "read_only", 1, 3) == "repeat_read", "只读调用可重读")
    c.ok(recovery_action("succeeded", "unsafe", 3, 3) == "reuse_result", "完成结果即使预算用尽也可复用")
    c.ok(recovery_action("failed", "read_only", 1, 3) == "reuse_result", "明确业务失败不无限重试")
    c.ok(recovery_action("in_flight", "idempotent", 3, 3) == "stop_budget", "单调用尝试数有上限")
    c.raises(ContractError, lambda: recovery_action("lost", "unsafe", 1, 2), "未知状态拒绝")
    c.raises(ContractError, lambda: recovery_action("planned", "unsafe", True, 2), "布尔值不是次数")
    with tempfile.TemporaryDirectory(prefix="lesson44_checks_") as root:
        root = Path(root)
        # 使用11个独立会话验证核心恢复矩阵。
        suite = scenario_suite(root / "cases")
        normal = suite["uninterrupted"]["after"]
        c.ok(normal["phase"] == "final_draft", "正常完成形成草稿")
        c.ok(normal["decision_turns"] == 4 and normal["dispatch_reservations"] == 4, "正常计数4轮4次")
        c.ok(normal["ticket_effects"] == 1 and normal["ticket_requests"] == 1, "正常只创建一张模拟工单")
        for name in ("plan_saved", "marked_not_sent", "effect_without_ack", "result_saved", "partial_read_batch"):
            row = suite[name]
            c.ok(row["interrupted"], name + "确实到达故障点")
            c.ok(row["after"]["phase"] == "final_draft", name + "恢复可完成")
            c.ok(row["after"]["final"] == normal["final"], name + "草稿一致")
            c.ok(row["after"]["ticket_effects"] == 1, name + "没有重复副作用")
        row = suite["effect_without_ack"]
        c.ok(row["before"]["call_states"]["call_003_01"] == "in_flight", "副作用提交后本地仍是运行中")
        c.ok(row["before"]["ticket_effects"] == 1, "本地未记结果但服务端效果已存在")
        c.ok(row["after"]["ticket_requests"] == 2 and row["after"]["dispatch_reservations"] == 5, "两次请求一次效果且预算不归零")
        c.ok(suite["marked_not_sent"]["after"]["ticket_requests"] == 1, "发送前中断只到达服务一次")
        c.ok(suite["result_saved"]["after"]["dispatch_reservations"] == 4, "已保存结果不重复执行")
        c.ok(suite["partial_read_batch"]["before"]["phase"] == "waiting", "部分批次阻塞下一决策")
        c.ok(suite["partial_read_batch"]["after"]["service_requests"] == 4, "恢复只执行剩余读取")
        c.ok(suite["decision_not_saved"]["after"]["decision_turns"] == 5, "未提交决策的预算不返还")
        c.ok(suite["decision_not_saved"]["after"]["ticket_effects"] == 1, "未提交计划没有产生副作用")
        unsafe = suite["unsafe_effect_unknown"]
        c.ok(unsafe["after"]["phase"] == "needs_review", "非幂等未知结果暂停")
        c.ok(unsafe["after"]["ticket_requests"] == 1, "未知非幂等调用未自动重试")
        c.ok(unsafe["after_reconciliation"]["phase"] == "final_draft", "已提交回执核对后可继续")
        c.ok(unsafe["after_reconciliation"]["ticket_requests"] == 1, "核对回执没有再次写入")
        c.ok(suite["unsafe_mark_only"]["after"]["ticket_effects"] == 0, "运行中可能尚未产生效果")
        c.ok("reconciliation_refused" in suite["unsafe_mark_only"], "没有回执不能伪造成功")
        c.ok(suite["retry_budget"]["after"]["phase"] == "limit_reached", "单调用重试预算持久化")
        c.ok(suite["total_budget"]["after"]["phase"] == "limit_reached", "总体预算恢复不重置")
        # 完成会话重启是只读复用，不增加事件/请求。
        completed_dir = root / "cases" / "uninterrupted"
        a, b = read_session(completed_dir), resume_session(completed_dir)
        c.ok(a == b, "已完成会话重复恢复不改任何数据库状态")
        with RunLock(completed_dir):
            c.raises(BusyRun, lambda: resume_session(completed_dir), "同目录第二执行者被本机锁阻止")
        c.raises(ContractError, lambda: create_session(completed_dir), "不覆盖恢复目录")
        # 服务端幂等与冲突；事务中间失败不会只留工单不留回执。
        svc = SimulatedService(root / "provider.db", create="idempotent")
        try:
            args = {"title": "t", "body": "body"}
            call = {"id": "c1", "tool": "create_ticket", "args": args,
                    "args_hash": fingerprint({"tool": "create_ticket", "args": args}),
                    "key": operation_key("r", "c1", "create_ticket")}
            c.raises(SimulatedCrash, lambda: svc.dispatch(call, fail_inside_transaction=True), "服务端事务故障传播")
            c.ok(svc.summary()["ticket_effects"] == 0 and svc.summary()["request_count"] == 0, "服务端效果与请求同事务回滚")
            first, second = svc.dispatch(call), svc.dispatch(call)
            c.ok(first == second, "同键重试返回相同回执")
            c.ok(svc.summary()["ticket_effects"] == 1, "同键不重复创建工单")
            changed = copy.deepcopy(call)
            changed["args"]["body"] = "changed"
            changed["args_hash"] = fingerprint({"tool": "create_ticket", "args": changed["args"]})
            c.raises(ContractError, lambda: svc.dispatch(changed), "同键不同参数拒绝")
            call2 = copy.deepcopy(call)
            call2["id"] = "c2"
            call2["key"] = operation_key("r", "c2", "create_ticket")
            svc.dispatch(call2)
            c.ok(svc.summary()["ticket_effects"] == 2, "同参数新意图允许新效果")
        finally:
            svc.close()
        bad = exercise_receipt_demo()
        c.ok(bad["unsafe"]["effects"] == 2, "盲目重发非幂等操作确实会重复")
        c.ok(bad["idempotent"]["effects"] == 1, "幂等服务独立验证")
        # 本地快照事务、状态机与结果消息原子性。
        d = create_session(root / "atomic")
        st = AgentStore(d / "agent.sqlite3")
        provider = SimulatedService(d / "service.sqlite3")
        try:
            ctl = DurableController(st, provider)
            v0, s0 = st.load()
            s1 = copy.deepcopy(s0); s1["phase"] = "planning"; s1["model_turns"] = 1
            c.raises(SimulatedCrash, lambda: st.commit(v0, s1, "test", fail_before_event=True), "快照事务内故障传播")
            c.ok(st.load() == (v0, s0) and len(st.events()) == 1, "快照与事件同时回滚")
            ctl.reserve_decision()
            c.raises(VersionConflict, lambda: st.commit(v0, s0, "stale"), "拒绝过期快照覆盖")
            ctl.commit_action(demo_policy(st.load()[1]))
            c.raises(ContractError, ctl.reserve_decision, "未收齐结果不能再次决策")
            selected = ctl.start_call("call_001_01")
            result = provider.dispatch(selected)
            version, before = st.load()
            c.raises(SimulatedCrash, lambda: ctl.finish_call(selected["id"], result, fail_before_event=True), "提交结果中断可观察")
            c.ok(st.load() == (version, before), "完成标记与tool消息没有只写一半")
            ctl.finish_call(selected["id"], result)
            stable = st.load()
            ctl.finish_call(selected["id"], result)
            c.ok(st.load() == stable, "重复确认相同结果不会重复追加消息")
            altered = copy.deepcopy(result); altered["data"]["text"] = "伪造"
            c.raises(ContractError, lambda: ctl.finish_call(selected["id"], altered), "不同结果不能覆盖完成回执")
            _, valid = st.load()
            fake = copy.deepcopy(valid); fake["messages"] = [m for m in fake["messages"] if m["role"] != "tool"]
            c.raises(ContractError, lambda: validate_state(fake), "完成调用缺tool消息不合法")
            fake = copy.deepcopy(valid); fake["dispatches"] = 0
            c.raises(ContractError, lambda: validate_state(fake), "预算与逐调用次数必须一致")
            fake = copy.deepcopy(valid); fake["calls"][0]["args"]["doc_id"] = "backup"
            c.raises(ContractError, lambda: validate_state(fake), "参数篡改不能静默通过")
            fake = copy.deepcopy(valid); fake["contract"]["documents"] = "changed"; fake["contract_hash"] = fingerprint(fake["contract"])
            c.raises(ContractError, lambda: validate_state(fake), "资料版本变化拒绝恢复")
        finally:
            st.close(); provider.close()
        # 明确业务错误也是终结结果，不无限重试。
        d = create_session(root / "business_error")
        st, provider = AgentStore(d / "agent.sqlite3"), SimulatedService(d / "service.sqlite3")
        try:
            ctl = DurableController(st, provider)
            ctl.reserve_decision()
            ctl.commit_action({"kind": "tools", "calls": [{"name": "read_doc", "arguments": {"doc_id": "missing"}}]})
            item = ctl.start_call("call_001_01")
            response = provider.dispatch(item)
            ctl.finish_call(item["id"], response)
            c.ok(st.load()[1]["calls"][0]["state"] == "failed", "业务失败有明确完成记录")
            c.ok(st.load()[1]["phase"] == "ready", "业务失败收齐结果后策略可决定下一步")
            c.raises(ContractError, lambda: ctl.start_call(item["id"]), "失败记录不被当成待执行")
        finally:
            st.close(); provider.close()
        # 服务丢失/替换不能建空账本后继续。
        missing = create_session(root / "missing_service")
        (missing / "service.sqlite3").unlink()
        c.raises(ContractError, lambda: resume_session(missing), "服务账本丢失拒绝恢复")
        swapped = create_session(root / "swapped_service")
        (swapped / "service.sqlite3").unlink()
        svc2 = SimulatedService(swapped / "service.sqlite3", create="idempotent"); svc2.close()
        c.raises(ContractError, lambda: resume_session(swapped), "服务身份更换拒绝恢复")
    exercise_counts = exercise_checks(operation_key_reference, recovery_action_reference, atomic_result_reference)
    c.ok(exercise_counts == {"A": 6, "B": 8, "C": 6}, "三组独立参考答案通过")
    return {"count": len(c.names), "passed": c.names, "exercise_counts": exercise_counts,
            "scope": "标准库本地模拟；不是模型规划、真实外部工具或断电测试"}

In [33]:
CHECK_REPORT = run_checks()
print("固定检查通过：", CHECK_REPORT["count"], "项")
print("参考答案检查：", CHECK_REPORT["exercise_counts"])
write_json(WORK_DIR / "checks.json", CHECK_REPORT)
if RUN_FULL_SUITE:
    write_json(FULL_RUN / "checks.json", CHECK_REPORT)

固定检查通过： 88 项
参考答案检查： {'A': 6, 'B': 8, 'C': 6}


## 17. 终端实操：真正退出一个进程再恢复

以下命令在终端运行，**不是把 `os._exit()` 粘贴到 Notebook 内核**。

先仅运行检查：

```bash
python lesson_44_agent_persistence_and_recovery.py --checks-only
```

创建新会话，故意在服务提交模拟工单后结束：

```bash
python lesson_44_agent_persistence_and_recovery.py --start my_run_44 --crash-at after_service_commit --crash-tool create_ticket
```

退出码 75 表示已到达指定教学中断点，不表示数据库需要删除。目录已经存在时不要重跑 `--start`。

只查看，不执行工具：

```bash
python lesson_44_agent_persistence_and_recovery.py --show my_run_44
```

恢复同一会话：

```bash
python lesson_44_agent_persistence_and_recovery.py --resume my_run_44
```

更接近进程突然退出的测试：给第一条 `--start` 命令额外添加 `--hard-crash`，并换一个尚不存在的目录。它用 `os._exit(75)` 跳过 Python 清理，但**仍不等于断电、磁盘故障或真实网络延迟测试**。

测试非幂等模拟服务：

```bash
python lesson_44_agent_persistence_and_recovery.py --start my_unsafe_44 --provider-mode unsafe --crash-at after_service_commit --crash-tool create_ticket
python lesson_44_agent_persistence_and_recovery.py --resume my_unsafe_44
python lesson_44_agent_persistence_and_recovery.py --reconcile my_unsafe_44 --call-id call_003_01
python lesson_44_agent_persistence_and_recovery.py --resume my_unsafe_44
```

第二条命令停在 `needs_review`；第三条只读取模拟服务的已提交记录，第四条才继续形成草稿。恢复使用已保存的服务模式与预算，不重设它们。

## 18. 本课没有解决的边界

- 没有加载真实语言模型，没有推理性能数据；固定策略不等于自主 Agent 规划。
- 没有实现任意远程工具的“恰好一次”。服务端幂等需要其保存回执、绑定参数、明确键的范围与保留期限；本课模拟回执不设置过期。
- 没有硬超时、分布式锁、进程沙箱、生产写操作审批和凭证管理。没有指数退避；只在受控恢复时进行有限重试。
- 不确定结果先暂停可能牺牲完成率。特别是发送前中断的非幂等操作，虽然实际没发生效果，本地也不能凭空证明安全重发。
- 同一机器锁和 SQLite 验证在 Linux 制作环境实测；Windows 分支未实际运行。
- 资料、策略、工具或服务身份变更时拒绝恢复；生产系统需要明确的迁移规则，不应直接修改数据库绕过检查。
- `final_draft` 是控制流程的结束，不是对资料支持、单位选择或实际业务结果的全面认证。

**记住：恢复不是重新做一遍；重试不是确认失败；保存了记录也不等于拥有跨系统事务。**

## 19. 参考资料与下一课

[Python sqlite3：事务与参数绑定](https://docs.python.org/3.12/library/sqlite3.html)  
[SQLite：原子提交及硬件假设](https://www.sqlite.org/atomiccommit.html)  
[AWS：让重试具备幂等性](https://aws.amazon.com/builders-library/making-retries-safe-with-idempotent-APIs/)  
[AWS：限制重试与避免重复副作用](https://docs.aws.amazon.com/wellarchitected/latest/framework/rel_mitigate_interaction_failure_limit_retries.html)  
[LangGraph：检查点与长期存储](https://docs.langchain.com/oss/python/langgraph/persistence)  
[Python：POSIX 文件锁](https://docs.python.org/3/library/fcntl.html) · [Windows 文件锁](https://docs.python.org/3/library/msvcrt.html)

下一课：**第 45 课 · Agent 权限边界、人工审批与受限代码执行**。先把模型建议、批准范围与实际执行参数绑定起来，再研究如何安全地把代码测试作为工具，而不是让模型获得任意 Shell 权限。